In [0]:
customers = spark.read.csv("/Workspace/Users/iluqmanshaik255@gmail.com/Drafts/customers_phase3.csv" , inferSchema= True , header = True)

sales = spark.read.csv("/Workspace/Users/iluqmanshaik255@gmail.com/Drafts/sales_phase3.csv" , inferSchema = True , header = True)

In [0]:
customers.printSchema()
customers.show()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- email: string (nullable = true)

+-----------+-------------+-------------+--------------+----------+-----------------+
|customer_id|customer_name|         city|         state|     phone|            email|
+-----------+-------------+-------------+--------------+----------+-----------------+
|        101|        Alice|      Chennai|    Tamil Nadu|9876543210|  alice@gmail.com|
|        102|          Bob|    Bengaluru|     Karnataka|9876543211|    bob@gmail.com|
|        103|      Charlie|    Hyderabad|     Telangana|9876543212|charlie@gmail.com|
|        104|        David|       Mumbai|   Maharashtra|9876543213|  david@gmail.com|
|        105|          Eva|         Pune|   Maharashtra|9876543214|    eva@gmail.com|
|        106|        Frank|        Delhi|         Delhi|9876543215|  fra

In [0]:
sales.printSchema()
sales.show()

root
 |-- sale_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- total_amount: integer (nullable = true)

+-------+-----------+----------+--------+--------+----------+------------+
|sale_id|customer_id| sale_date| product|quantity|unit_price|total_amount|
+-------+-----------+----------+--------+--------+----------+------------+
|   S001|        101|2026-07-01|  Laptop|       1|     65000|       65000|
|   S002|        102|2026-07-01|   Phone|       2|     25000|       50000|
|   S003|        103|2026-07-01|Keyboard|       3|      1500|        4500|
|   S004|        104|2026-07-02| Monitor|       1|     18000|       18000|
|   S005|        105|2026-07-02|   Mouse|       2|       800|        1600|
|   S006|        106|2026-07-02|  Tablet|       1|     30000|       30000|
|   S007|        107|2026-07-

In [0]:
print("Total Rows: " ,customers.count())
print("Total Columns: ", len(customers.columns))

Total Rows:  50
Total Columns:  6


In [0]:
print("Total Rows: " , sales.count())
print("Total Columns: " , len(sales.columns))

Total Rows:  60
Total Columns:  7


In [0]:
from pyspark.sql.functions import col

In [0]:
for column in customers.columns:
    null_count = customers.filter(col(column).isNull()).count()
    print(f"{column} : {null_count}")

customer_id : 2
customer_name : 0
city : 0
state : 0
phone : 0
email : 0


In [0]:
for column in sales.columns:
    null_count = sales.filter(col(column).isNull()).count()
    print(f"{column} : {null_count}")

sale_id : 0
customer_id : 2
sale_date : 0
product : 0
quantity : 0
unit_price : 0
total_amount : 0


In [0]:
print("customers Before dropping NUll: ", customers.count())
customers = customers.dropna(subset=["customer_id"])
print("Customers After dropping Null: " , customers.count())

customers Before dropping NUll:  50
Customers After dropping Null:  48


In [0]:
print("Sales Before dropping Null:", sales.count())
sales = sales.dropna(subset = ["customer_id"])
print("Sales After dropping Null: " ,sales.count())

Sales Before dropping Null: 60
Sales After dropping Null:  58


In [0]:
customers.createOrReplaceTempView("customers")
sales.createOrReplaceTempView("sales")

In [0]:
print("Duplicate Customers: " , customers.count() - customers.distinct().count())

Duplicate Customers:  0


In [0]:
print("Duplicate Sales: " , sales.count() - sales.distinct().count())

Duplicate Sales:  0


In [0]:
customers = customers.dropDuplicates()
sales = sales.dropDuplicates()

In [0]:
%sql 
-- Calculate the total amount spent by each customer
SELECT c.customer_name ,
SUM(s.total_amount) AS total_amount_spent
FROM customers c   
INNER JOIN sales s   
ON c.customer_id = s.customer_id
GROUP BY c.customer_id , c.customer_name
ORDER BY total_amount_spent DESC;

customer_name,total_amount_spent
Sophia,130000
Nina,130000
Bob,115000
David,107000
Frank,95000
Ian,75000
Alice,72000
Hannah,65000
Meera,65000
Irene,65000


In [0]:
from pyspark.sql.functions import sum
customers.join(sales ,
            on = "customer_id" ,
            how = "inner") \
            .groupBy("customer_id" ,"customer_name") \
            .agg(sum("total_amount").alias("total_amount_spent")) \
             .orderBy("total_amount_spent" , ascending = False) \
             .show()      

+-----------+-------------+------------------+
|customer_id|customer_name|total_amount_spent|
+-----------+-------------+------------------+
|        119|       Sophia|            130000|
|        140|         Nina|            130000|
|        102|          Bob|            115000|
|        104|        David|            107000|
|        106|        Frank|             95000|
|        135|          Ian|             75000|
|        101|        Alice|             72000|
|        134|       Hannah|             65000|
|        109|        Irene|             65000|
|        145|        Tarun|             65000|
|        130|        Diana|             65000|
|        125|         Yash|             65000|
|        150|        Meera|             65000|
|        133|       George|             60000|
|        111|        Kevin|             60000|
|        147|        Akash|             60000|
|        103|      Charlie|             59500|
|        127|        Aaron|             50000|
|        116|

In [0]:
%sql
-- Display the top 3 customers based on total amount spent
SELECT c.customer_id ,
c.customer_name,
SUM(s.total_amount) AS total_amount_spent
FROM customers c   
INNER JOIN sales s   
ON c.customer_id = s.customer_id
GROUP BY c.customer_id , c.customer_name
ORDER BY total_amount_spent DESC
LIMIT 3;

customer_id,customer_name,total_amount_spent
119,Sophia,130000
140,Nina,130000
102,Bob,115000


In [0]:
from pyspark.sql.functions import sum
customers.join(sales , on = "customer_id" , how = "inner") \
    .groupBy("customer_id" , "customer_name" ) \
        .agg(sum("total_amount").alias("total_amount_spent")) \
        .orderBy("total_amount_spent" , ascending = False) \
        .limit(3) \
        .show()

+-----------+-------------+------------------+
|customer_id|customer_name|total_amount_spent|
+-----------+-------------+------------------+
|        140|         Nina|            130000|
|        119|       Sophia|            130000|
|        102|          Bob|            115000|
+-----------+-------------+------------------+



In [0]:
%sql
--Find customers who have never placed an order
SELECT * FROM customers
WHERE customer_id NOT IN (
    SELECT customer_id 
    FROM sales  
);

customer_id,customer_name,city,state,phone,email


In [0]:
customers.join(sales, 
               on = "customer_id",
               how = "left_anti").show()

+-----------+-------------+----+-----+-----+-----+
|customer_id|customer_name|city|state|phone|email|
+-----------+-------------+----+-----+-----+-----+
+-----------+-------------+----+-----+-----+-----+



In [0]:
%sql
-- Calculate the total revenue generated from each city.
SELECT c.city , 
SUM(s.total_amount) AS total_revenue
FROM customers c  
INNER JOIN sales s    
ON c.customer_id = s.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC;

city,total_revenue
Visakhapatnam,303000
Kolkata,275000
Pune,250800
Delhi,193000
Mumbai,185500
Bengaluru,169000
Kochi,158000
Chennai,156600
Hyderabad,147400
Jaipur,83000


In [0]:
from pyspark.sql.functions import sum
customers.join(sales , on = "customer_id" , how = "inner") \
    .groupBy("city") \
    .agg(sum("total_amount").alias("total_revenue")) \
    .orderBy("total_revenue" , ascending = False) \
    .show()

+-------------+-------------+
|         city|total_revenue|
+-------------+-------------+
|Visakhapatnam|       303000|
|      Kolkata|       275000|
|         Pune|       250800|
|        Delhi|       193000|
|       Mumbai|       185500|
|    Bengaluru|       169000|
|        Kochi|       158000|
|      Chennai|       156600|
|    Hyderabad|       147400|
|       Jaipur|        83000|
+-------------+-------------+



In [0]:
from pyspark.sql.functions import avg
customers.join(sales , on ="customer_id" , how = "inner") \
    .groupBy("customer_id" , "customer_name") \
        .agg(avg("total_amount").alias("average_amount_spent")) \
            .orderBy("average_amount_spent" , ascending = False) \
            .show()

customer_id,customer_name,average_amount_spent
119,Sophia,130000.0
140,Nina,130000.0
135,Ian,75000.0
134,Hannah,65000.0
150,Meera,65000.0
109,Irene,65000.0
125,Yash,65000.0
130,Diana,65000.0
145,Tarun,65000.0
147,Akash,60000.0


In [0]:
from pyspark.sql.functions import avg
customers.join(sales , on ="customer_id" , how = "inner") \
    .groupBy("customer_id" , "customer_name") \
        .agg(avg("total_amount").alias("average_amount_spent")) \
            .orderBy("average_amount_spent" , ascending = False) \
            .show()

+-----------+-------------+--------------------+
|customer_id|customer_name|average_amount_spent|
+-----------+-------------+--------------------+
|        140|         Nina|            130000.0|
|        119|       Sophia|            130000.0|
|        135|          Ian|             75000.0|
|        150|        Meera|             65000.0|
|        134|       Hannah|             65000.0|
|        130|        Diana|             65000.0|
|        125|         Yash|             65000.0|
|        109|        Irene|             65000.0|
|        145|        Tarun|             65000.0|
|        133|       George|             60000.0|
|        147|        Akash|             60000.0|
|        111|        Kevin|             60000.0|
|        102|          Bob|             57500.0|
|        116|        Peter|             50000.0|
|        127|        Aaron|             50000.0|
|        149|        Rohit|             50000.0|
|        106|        Frank|             47500.0|
|        104|       

In [0]:
%sql
-- Display customers who have placed more than one order.
SELECT c.customer_id , c.customer_name,
COUNT(s.sale_id) AS orders_placed
FROM customers c   
INNER JOIN sales s   
ON c.customer_id = s.customer_id 
GROUP BY c.customer_id , c.customer_name 
HAVING COUNT(s.sale_id) > 1
ORDER BY c.customer_id;

customer_id,customer_name,orders_placed
101,Alice,3
102,Bob,2
103,Charlie,3
104,David,3
105,Eva,3
106,Frank,2


In [0]:
from pyspark.sql.functions import count
customers.join(sales , on = "customer_id" , how = "inner") \
    .groupBy("customer_id" , "customer_name") \
    .agg(count("sale_id").alias("orders_placed")) \
    .where("orders_placed > 1") \
    .orderBy("customer_id" , ascending = True) \
    .show()

+-----------+-------------+-------------+
|customer_id|customer_name|orders_placed|
+-----------+-------------+-------------+
|        101|        Alice|            3|
|        102|          Bob|            2|
|        103|      Charlie|            3|
|        104|        David|            3|
|        105|          Eva|            3|
|        106|        Frank|            2|
+-----------+-------------+-------------+

